In [1]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from qdrant_client import QdrantClient, models
from dotenv import load_dotenv
from tqdm.auto import tqdm
import os

load_dotenv()

2026-01-15 00:13:08.165288765 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


True

In [2]:
class Config:
    GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY") 
    QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
    COLLECTION_NAME = "knowledgebase_collection"
    EMBEDDING_MODEL = "models/gemini-embedding-001"

In [3]:
client = QdrantClient(url=Config.QDRANT_URL)

embeddings = GoogleGenerativeAIEmbeddings(
    api_key=Config.GOOGLE_API_KEY,
    model=Config.EMBEDDING_MODEL
)

In [4]:
all_docs = []

## Load Product Documents

In [5]:
docs_path = Path("../data/knowledge_base/products/")
existing_files = {f.name for f in docs_path.glob("*.md")}

if not existing_files:
    raise FileNotFoundError(f"Missing required documents: {existing_files}")

for file_path in docs_path.glob("*.md"):
    group = file_path.stem  
    loader = TextLoader(str(file_path))
    doc = loader.load()[0]
    splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "product_name")])
    product_docs = splitter.split_text(doc.page_content)
    
    for pd in product_docs:
        all_docs.append(
            Document(
                page_content=pd.page_content,
                metadata={
                    "doc_type": "product",
                    "product_name": pd.metadata["product_name"],
                    "group": group
                }
            )
        )

## Load Plant Information Documents

In [6]:
plants_path = Path("../data/knowledge_base/plants/plants.md")
if plants_path.exists():
    loader = TextLoader(str(plants_path))
    plants_doc = loader.load()[0]
   
    # Split by plant (h1) and sections (h2, h3, h4)
    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "plant_name"),
            ("##", "section"),
            ("###", "subsection"),
            ("####", "disease_name")
        ]
    )
    plant_chunks = splitter.split_text(plants_doc.page_content)
   
    for chunk in plant_chunks:
        # Skip empty content chunks
        if not chunk.page_content.strip():
            continue
           
        # Extract metadata
        plant_name = chunk.metadata.get("plant_name", "Unknown").lower()
        section = chunk.metadata.get("section", "General")
        subsection = chunk.metadata.get("subsection", "")
        disease_name = chunk.metadata.get("disease_name", "")
       
        # Create meaningful section identifier
        section_id = section
        if subsection:
            section_id = f"{section} - {subsection}"
        if disease_name:
            section_id = f"{section} - {disease_name}"
       
        all_docs.append(
            Document(
                page_content=chunk.page_content,
                metadata={
                    "doc_type": "plant_info",
                    "plant_name": plant_name,
                    "section": section_id,
                    "content_type": section.lower().replace(" ", "_")
                }
            )
        )

## Document Summary

In [7]:
print(f"Total documents loaded: {len(all_docs)}")
print("\nDocument types:")
products = [d for d in all_docs if d.metadata["doc_type"] == "product"]
plants = [d for d in all_docs if d.metadata["doc_type"] == "plant_info"]
print(f"Products: {len(products)}")
print(f"Plant info: {len(plants)}")

Total documents loaded: 2026

Document types:
Products: 95
Plant info: 1931


In [8]:
plants[0]

Document(metadata={'doc_type': 'plant_info', 'plant_name': 'african eggplant', 'section': 'Description', 'content_type': 'description'}, page_content='African eggplant, Solanum aethiopicum , is a deciduous shrub in the family Solanaceae which is grown for its edible fruits which are cooked and eaten as a vegetable. African eggplant is a highly branching plant which can grow up to 2 m (6.6 ft) in height. The leaves of the plant are arranged alternately on the stems and have smooth or lobed margins. Leaf blades may reach up to 30 cm (11.8 in) in length and 21 cm (8.3 in) in width. The leaf petioles are oval or elliptical in shape, reaching up to 11 cm (4.3 in) in length. Plants produce clusters of up to 12 white flowers which develop into egg or spindle-shaped berries which are red to orange in color with a smooth or grooved surface depending on variety. African eggplant may also be referred to as scarlet eggplant, bitter tomato, mock tomato, garden egg or Ethiopian nightshade and is nat

## Create Qdrant Vector Store

In [9]:
# Recreate collection (deletes existing if present)
try:
    client.delete_collection(collection_name=Config.COLLECTION_NAME)
    print(f"Deleted existing collection: {Config.COLLECTION_NAME}")
except Exception as e:
    print(f"Collection {Config.COLLECTION_NAME} doesn't exist yet: {e}")

# For hybrid search, we need both dense and sparse vector configurations
vector_size = 3072 
client.create_collection(
    collection_name=Config.COLLECTION_NAME,
    vectors_config={
        "dense": models.VectorParams(
            size=vector_size,
            distance=models.Distance.COSINE
        )
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams()
    }
)

Collection knowledgebase_collection doesn't exist yet: [Errno 104] Connection reset by peer


ResponseHandlingException: [Errno 104] Connection reset by peer

## Create Full-Text Indexes for Metadata Filtering

Create full-text indexes on `metadata.plant_name` and `metadata.section` to enable case-insensitive filtering with `MatchText`.

In [ ]:
# Create full-text indexes for case-insensitive filtering
client.create_payload_index(
    collection_name=Config.COLLECTION_NAME,
    field_name="metadata.plant_name",
    field_schema=models.TextIndexParams(
        type=models.TextIndexType.TEXT,
        tokenizer=models.TokenizerType.WORD,
        min_token_len=2,
        max_token_len=20,
        lowercase=True
    )
)

client.create_payload_index(
    collection_name=Config.COLLECTION_NAME,
    field_name="metadata.section",
    field_schema=models.TextIndexParams(
        type=models.TextIndexType.TEXT,
        tokenizer=models.TokenizerType.WORD,
        min_token_len=2,
        max_token_len=20,
        lowercase=True
    )
)

print("✓ Created full-text indexes:")
print("  - metadata.plant_name (lowercase word tokenization)")
print("  - metadata.section (lowercase word tokenization)")

✓ Created full-text indexes:
  - metadata.plant_name (lowercase word tokenization)
  - metadata.section (lowercase word tokenization)


## Initialize Sparse Embeddings (SPLADE)

SPLADE provides learned sparse representations for better keyword matching than BM25.

In [ ]:
# Initialize SPLADE sparse embeddings
sparse_embeddings = FastEmbedSparse(model_name="prithivida/Splade_PP_en_v1")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/755 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

model.onnx:   0%|          | 0.00/532M [00:00<?, ?B/s]

## Ingest Hybrid Vectors with Progress Bar

Using Qdrant client directly to ingest both dense and sparse vectors with a progress bar.

In [ ]:
batch_size = 50
points = []

print("Generating dense and sparse embeddings...")
for idx, doc in enumerate(tqdm(all_docs, desc="Processing documents")):
    # Generate dense embedding (Gemini)
    dense_embedding = embeddings.embed_query(doc.page_content)
    
    sparse_embedding = sparse_embeddings.embed_query(doc.page_content)
    
    # Create point with both dense and sparse vectors
    point = models.PointStruct(
        id=idx,
        vector={
            "dense": dense_embedding,
            "sparse": models.SparseVector(
                indices=sparse_embedding.indices,
                values=sparse_embedding.values
            )
        },
        payload={
            "page_content": doc.page_content,
            "metadata": doc.metadata
        }
    )
    points.append(point)
    
    # Upload in batches
    if len(points) >= batch_size:
        client.upsert(
            collection_name=Config.COLLECTION_NAME,
            points=points
        )
        points = []

# Upload remaining points
if points:
    client.upsert(
        collection_name=Config.COLLECTION_NAME,
        points=points
    )

print(f"Uploaded {len(all_docs)} documents to collection '{Config.COLLECTION_NAME}'")

Generating dense and sparse embeddings...


Processing documents:   0%|          | 0/2026 [00:00<?, ?it/s]

Uploaded 2026 documents to collection 'knowledgebase_collection'


## Create Hybrid Retriever from Existing Collection

Now create a LangChain retriever with hybrid search mode from the existing collection.

In [ ]:
from langchain_qdrant import RetrievalMode

# Create LangChain vector store from existing Qdrant collection with hybrid mode
vector_store = QdrantVectorStore(
    client=client,
    collection_name=Config.COLLECTION_NAME,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",  # Name of dense vectors in collection
    sparse_vector_name="sparse",  # Name of sparse vectors in collection
)

print(f"✓ Created hybrid retriever from collection '{Config.COLLECTION_NAME}'")
print(f"  - Retrieval mode: HYBRID (Dense + Sparse)")
print(f"  - Dense: Gemini embeddings")
print(f"  - Sparse: SPLADE embeddings")

✓ Created hybrid retriever from collection 'knowledgebase_collection'
  - Retrieval mode: HYBRID (Dense + Sparse)
  - Dense: Gemini embeddings
  - Sparse: SPLADE embeddings


## Test Hybrid Search

Test the hybrid search (combines dense semantic + sparse keyword matching).

In [ ]:
test_queries = [
    "apple leaf spot disease symptoms",  
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"Mode: HYBRID (Dense Gemini + Sparse SPLADE)")
    print(f"{'='*60}")
   
    # Hybrid similarity search (top 10 results)
    results = vector_store.similarity_search(query, k=10)
   
    for i, doc in enumerate(results, 1):
        doc_type = doc.metadata['doc_type']
        print(f"\nResult {i} ({doc_type}):")
       
        if doc_type == "product":
            print(f"Product: {doc.metadata['product_name']}")
            print(f"Group: {doc.metadata['group']}")
        else:
            print(f"Plant: {doc.metadata['plant_name']}")
            print(f"Section: {doc.metadata['section']}")
       
        print(f"Content: {doc.page_content[:300]}...")


Query: apple leaf spot disease symptoms
Mode: HYBRID (Dense Gemini + Sparse SPLADE)

Result 1 (plant_info):
Plant: vanilla
Section: Diseases - Anthracnose Colletotrichum spp.
Content: - Symptoms: Small, sunken, dark brown spots on leaves, fruits, stems and/or flowers; infected fruits dropping from plants before they reach maturity; damage to fruit is more pronounced during warm and humid periods of the growing season; symptoms generally develop first on apical parts of plant and ...

Result 2 (plant_info):
Plant: cherry (including sour)
Section: Diseases - Cherry leaf spot Coccomyces hiemalis
Content: - Symptoms: Small, red-purple spots on upper surfaces of leaves which turn brown and may coalesce; leaves may become chlorotic if there are a few lesions present; if tree becomes severely defoliated fruit may fail to develop properly and remain light in color and watery in texture  
- Cause: Fungus ...

Result 3 (plant_info):
Plant: cucumber
Section: Diseases - Bacterial leaf spot Xantho

## Search Functions with Metadata Filters

In [ ]:
def search_plant_info(query, k=3):
    """Search plant information only using metadata filter"""
    qdrant_filter = models.Filter(
        must=[
            models.FieldCondition(
                key="metadata.doc_type",
                match=models.MatchValue(value="plant_info"),
            )
        ]
    )
    return vector_store.similarity_search(query, k=k, filter=qdrant_filter)

def search_products(query, k=3):
    """Search products only using metadata filter"""
    qdrant_filter = models.Filter(
        must=[
            models.FieldCondition(
                key="metadata.doc_type",
                match=models.MatchValue(value="product"),
            )
        ]
    )
    results = vector_store.similarity_search(query, k=k, filter=qdrant_filter)
    return [
        {"content": doc.page_content, "metadata": doc.metadata}
        for doc in results
    ]

def search_disease_info(query, plant_name=None, k=3):
    """Search disease-specific info using metadata filter"""
    must_conditions = [
        models.FieldCondition(
            key="metadata.doc_type",
            match=models.MatchValue(value="plant_info"),
        )
    ]
    
    if plant_name:
        must_conditions.append(
            models.FieldCondition(
                key="metadata.plant_name",
                match=models.MatchValue(value=plant_name),
            )
        )
    
    qdrant_filter = models.Filter(must=must_conditions)
    return vector_store.similarity_search(query, k=k, filter=qdrant_filter)

def search_products_by_group(query, group_name, k=3):
    """Search products by specific group using metadata filter"""
    qdrant_filter = models.Filter(
        must=[
            models.FieldCondition(
                key="metadata.doc_type",
                match=models.MatchValue(value="product"),
            ),
            models.FieldCondition(
                key="metadata.group",
                match=models.MatchValue(value=group_name),
            )
        ]
    )
    return vector_store.similarity_search(query, k=k, filter=qdrant_filter)

## Test Product Search

In [ ]:
print(f"\n{'='*60}")
print("Product Search Test")
print(f"{'='*60}")

test_queries = [
    "Insektisida untuk tanaman tomat",
]

for query in test_queries:
    print(f"\nQuery: {query}")
    product_results = search_products(query, k=2)
    
    for i, doc in enumerate(product_results, 1):
        print(f"\nResult {i}:")
        print(f"Product Name: {doc['metadata'].get('product_name', 'N/A')}")
        print(f"Group: {doc['metadata'].get('group', 'N/A')}")
        print(f"Content: {doc['content'][:300]}...")


Product Search Test

Query: Insektisida untuk tanaman tomat



Result 1:
Product Name: KON UP 76 SG
Group: herbicide
Content: No image available  
**Active Ingredient:** Monoammo glifosat 75.7  
**Description:** Herbicide sitemik purnah tumbuh in granule form that can terdespersi dalam air yellow kebiruan to control weeds alang – alang ( Imperata Cyliandrica ) lahan tanpa tanaman.  
**Application:** High-volume spraying de...

Result 2:
Product Name: BIONIK 400 EC
Group: insecticide
Content: ![Image of product: BIONIK 400 EC](https://kresna.co.id/sarikresnakimia/wp-content/uploads/2014/11/Bionik-400-EC-1000x1000--360x360.png)  
**Active Ingredient:** Dimetoate 400 g/l  
**Description:** Insecticide poison contact and stomach as a concentrate colored kekuning-kuningan that can be emulsif...


## Reranking with Jina AI

Reranking improves retrieval quality by reordering initial results using a more sophisticated model. We'll use Jina AI's reranker with contextual compression.

In [ ]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors import JinaRerank

# Get Jina AI API key from environment
JINA_API_KEY = os.getenv("JINA_API_KEY")

### Setup Reranking Retriever

Create a retriever that first fetches many candidates (k=20), then reranks to get the top-k most relevant results.

In [ ]:
base_retriever = vector_store.as_retriever(search_kwargs={"k": 20})

compressor = JinaRerank(
    model="jina-reranker-v3", 
    jina_api_key=JINA_API_KEY,
    top_n=5
)

# Create compression retriever with reranking
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

### Test Reranked Results

Compare hybrid search results with and without reranking to see quality improvements.

In [ ]:

test_query = "grape leaves with numerous small, circular, light brown spots."

print(f"Query: {test_query}\n")

# Get results with reranking
print("="*60)
print("HYBRID SEARCH + JINA RERANKING")
print("="*60)
reranked_results = rerank_retriever.invoke(test_query)

for i, doc in enumerate(reranked_results, 1):
    doc_type = doc.metadata['doc_type']
    print(f"\n{i}. ({doc_type})", end=" ")
    
    if doc_type == "product":
        print(f"- {doc.metadata['product_name']}")
    else:
        print(f"- {doc.metadata['plant_name']} / {doc.metadata['section']}")
    
    # JinaRerank typically adds relevance_score to metadata
    score = doc.metadata.get("relevance_score")
    if score:
        print(f"   Score: {score:.4f}")
    
    print(f"   Content: {doc.page_content[:200]}...")

# Compare with hybrid search without reranking
print("\n" + "="*60)
print("HYBRID SEARCH ONLY (no reranking)")
print("="*60)
hybrid_only = vector_store.similarity_search(test_query, k=5)

for i, doc in enumerate(hybrid_only, 1):
    doc_type = doc.metadata['doc_type']
    print(f"\n{i}. ({doc_type})", end=" ")
    
    if doc_type == "product":
        print(f"- {doc.metadata['product_name']}")
    else:
        print(f"- {doc.metadata['plant_name']} / {doc.metadata['section']}")
    
    print(f"   Content: {doc.page_content[:200]}...")


Query: grape leaves with numerous small, circular, light brown spots.

HYBRID SEARCH + VOYAGE AI RERANKING

1. (plant_info) - almond / Diseases - Alternaria leaf spot Alternaria alternata
   Content: - Symptoms: Light brown lesions on leaves which expand to form circular lesions on leaf blade or semi-circular lesions on margin; leaves may develop light yellow necrosis which dries and turns tan in ...

2. (plant_info) - grape / Diseases - Black rot Guignardia bidwellii
   Content: - Symptoms: Brown lesions on the leaves that develop black dots (pycnidia); grapes have light spots that eventually form pycnidia; grapes harden and turn black, while still remaining on the vine  
- C...

3. (plant_info) - peanut (groundnut) / Diseases - Phyllostica leaf spot Phyllostica arachidis-hypogaea
   Content: - Symptoms: Circular lesions with red-brown margins and light brown or tan centers on leaves; centers of lesions may dry out and drop from leaf resulting in a "shot-hole" appearance.  
- Cause: F

### Advanced: Reranking with Metadata Filters

Combine metadata filtering with hybrid search and reranking for even more precise results using Jina AI.

In [ ]:

def hybrid_search_with_rerank(query, doc_type=None, plant_name=None, k=5, fetch_k=20):
    """
    Hybrid search with optional metadata filters and Jina AI reranking
    
    Args:
        query: Search query
        doc_type: Filter by document type ('product' or 'plant_info')
        plant_name: Filter by plant name (only for plant_info docs)
        k: Number of final results to return after reranking
        fetch_k: Number of candidates to fetch before reranking
    """
    # Build filter conditions
    must_conditions = []
    if doc_type:
        must_conditions.append(
            models.FieldCondition(
                key="metadata.doc_type",
                match=models.MatchValue(value=doc_type),
            )
        )
    if plant_name:
        must_conditions.append(
            models.FieldCondition(
                key="metadata.plant_name",
                match=models.MatchValue(value=plant_name),
            )
        )
    
    # Create filtered retriever
    if must_conditions:
        qdrant_filter = models.Filter(must=must_conditions)
        base_retriever = vector_store.as_retriever(
            search_kwargs={"k": fetch_k, "filter": qdrant_filter}
        )
    else:
        base_retriever = vector_store.as_retriever(
            search_kwargs={"k": fetch_k}
        )
    
    # Setup reranker
    compressor = JinaRerank(
        model="jina-reranker-v3",
        jina_api_key=JINA_API_KEY,
        top_n=k
    )
    
    # Create compression retriever
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=base_retriever
    )
    
    return compression_retriever.invoke(query)


### Example: Search Products Only with Reranking

In [ ]:

query = "Insecticide for tomato plants"

print(f"Query: {query}")
print("Filter: doc_type='product'\n")

results = hybrid_search_with_rerank(
    query=query,
    doc_type="product",
    k=3,
    fetch_k=15
)

print("="*60)
print("TOP 3 PRODUCTS (Hybrid + Reranked)")
print("="*60)

for i, doc in enumerate(results, 1):
    print(f"\n{i}. {doc.metadata['product_name']}")
    print(f"   Group: {doc.metadata['group']}")
    print(f"   Content: {doc.page_content[:250]}...")


Query: Insecticide for tomato plants
Filter: doc_type='product'

TOP 3 PRODUCTS (Hybrid + Reranked)

1. KILIRI 20 EC
   Group: insecticide
   Content: ![Image of product: KILIRI 20 EC](https://kresna.co.id/sarikresnakimia/wp-content/uploads/2021/02/kiliri-360x360.png)  
**Active Ingredient:** Abamektin 20 g/l  
**Description:** Insecticide poison contact and stomach as a concentrate yellow muda tha...

2. TAMACRON 500 EC
   Group: insecticide
   Content: ![Image of product: TAMACRON 500 EC](https://kresna.co.id/sarikresnakimia/wp-content/uploads/2014/11/Tamacron-500-EC-1000x1000-360x360.png)  
**Active Ingredient:** Profenofos 500 g/l  
**Description:** Insecticide poison contact and stomach as an em...

3. OKRITE 20 EC
   Group: insecticide
   Content: ![Image of product: OKRITE 20 EC](https://kresna.co.id/sarikresnakimia/wp-content/uploads/2014/11/Okrite-20-EC-1000x1000-360x360.png)  
**Active Ingredient:** Abamectin 20 g/l  
**Description:** Insecticide poison contact and stomach as

In [ ]:

test_query = "grape leaves with numerous small, circular, light brown spots."

print(f"Query: {test_query}\n")

print("="*60)
print("PLANT INFO SEARCH (doc_type='plant_info') + RERANKING")
print("="*60)
results = hybrid_search_with_rerank(
    query=test_query,
    doc_type="plant_info",
    k=5,    
    fetch_k=20
)

for i, doc in enumerate(results, 1):
    print(f"\n{i}. {doc.metadata['plant_name']} / {doc.metadata['section']}")
    print(f"   Content: {doc.page_content[:200]}...")

Query: grape leaves with numerous small, circular, light brown spots.

PLANT INFO SEARCH (doc_type='plant_info') + RERANKING

1. grape / Diseases - Black rot Guignardia bidwellii
   Content: - Symptoms: Brown lesions on the leaves that develop black dots (pycnidia); grapes have light spots that eventually form pycnidia; grapes harden and turn black, while still remaining on the vine  
- C...

2. starfruit (carambola) / Diseases - Alternaria black spot (Brown spot) Alternaria alternata
   Content: - Symptoms: Small, circular light brown or black spots on skin of fruits; lesions develop sunken centres and olive-brown spores  
- Cause: Fungus  
- Comments: Fungi spread via wind and rain and enter...

3. almond / Diseases - Alternaria leaf spot Alternaria alternata
   Content: - Symptoms: Light brown lesions on leaves which expand to form circular lesions on leaf blade or semi-circular lesions on margin; leaves may develop light yellow necrosis which dries and turns tan in ...

4. peanut (

## Reranking with Jina Reranker v3 



In [ ]:
from langchain_community.document_compressors import JinaRerank

# Initialize Jina Reranker v3 
# This uses the Jina AI API with the model 'jina-reranker-v3'
jina_api_key = os.getenv("JINA_API_KEY")
jina_compressor = JinaRerank(
    model="jina-reranker-v3", 
    jina_api_key=jina_api_key,
    top_n=5
)

# Create compression retriever with Jina
jina_rerank_retriever = ContextualCompressionRetriever(
    base_compressor=jina_compressor,
    base_retriever=base_retriever
)

print("✓ Initialized Jina Reranker v3")

✓ Initialized Jina Reranker v3


In [ ]:
test_query = "grape leaves with numerous small, circular, light brown spots."

print(f"Query: {test_query}\n")

print("="*60)
print("HYBRID SEARCH + JINA RERANKER V3")
print("="*60)
jina_results = jina_rerank_retriever.invoke(test_query)

for i, doc in enumerate(jina_results, 1):
    doc_type = doc.metadata['doc_type']
    print(f"\n{i}. ({doc_type})", end=" ")
    
    if doc_type == "product":
        print(f"- {doc.metadata['product_name']}")
    else:
        print(f"- {doc.metadata['plant_name']} / {doc.metadata['section']}")
    
    # JinaRerank typically adds relevance_score to metadata
    score = doc.metadata.get("relevance_score")
    if score:
        print(f"   Score: {score:.4f}")
    
    print(f"   Content: {doc.page_content[:200]}...")


Query: grape leaves with numerous small, circular, light brown spots.

HYBRID SEARCH + JINA RERANKER V3

1. (plant_info) - grape / Diseases - Black rot Guignardia bidwellii
   Score: 0.2962
   Content: - Symptoms: Brown lesions on the leaves that develop black dots (pycnidia); grapes have light spots that eventually form pycnidia; grapes harden and turn black, while still remaining on the vine  
- C...

2. (plant_info) - almond / Diseases - Alternaria leaf spot Alternaria alternata
   Score: 0.1522
   Content: - Symptoms: Light brown lesions on leaves which expand to form circular lesions on leaf blade or semi-circular lesions on margin; leaves may develop light yellow necrosis which dries and turns tan in ...

3. (plant_info) - grape / Diseases - Esca (Black Measles or Spanish Measles) Phaeomoniella aleophilum, Phaeomoniella chlamydospora
   Score: 0.0940
   Content: - Symptoms: Symptom appears on leaves, trunk, canes and berries. On leaves we will see intervenaial striping looks like

## Local Inference with Jina Reranker v3 (Hugging Face)

If you have the model downloaded or want to run it locally using the `transformers` library, you can use the following implementation. This avoids API calls but requires more local resources.


In [ ]:
from transformers import AutoModel
import torch

# Load the model locally
# Note: This requires 'transformers' and 'einops' installed
try:
    model = AutoModel.from_pretrained(
        'jinaai/jina-reranker-v3', 
        dtype="auto", 
        trust_remote_code=True
    )
    model.eval()
    if torch.cuda.is_available():
        model.to("cuda")
    print("✓ Successfully loaded Jina Reranker v3 locally")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Make sure you have 'transformers' and 'einops' installed.")

def local_jina_rerank(query, documents, top_n=5):
    """
    Rerank documents locally using Jina Reranker v3
    """
    doc_texts = [doc.page_content for doc in documents]
    
    # model.rerank returns a list of dictionaries
    results = model.rerank(query, doc_texts, top_n=top_n)
    
    reranked_docs = []
    for res in results:
        doc = documents[res['index']]
        # Add score to metadata
        doc.metadata['relevance_score'] = res['relevance_score']
        reranked_docs.append(doc)
        
    return reranked_docs


In [ ]:
# Test Local Jina Reranker
if 'model' in locals():
    # Get base results first
    base_docs = base_retriever.invoke(test_query)
    
    print("="*60)
    print("HYBRID SEARCH + LOCAL JINA RERANKER V3")
    print("="*60)
    
    local_results = local_jina_rerank(test_query, base_docs, top_n=5)
    
    for i, doc in enumerate(local_results, 1):
        doc_type = doc.metadata['doc_type']
        print(f"\n{i}. ({doc_type}) - {doc.metadata.get('plant_name', doc.metadata.get('product_name'))}")
        print(f"   Score: {doc.metadata['relevance_score']:.4f}")
        print(f"   Content: {doc.page_content[:200]}...")
